In [1]:
import torch
import pandas as pd

from metrics import calculate_apd, calculate_prt, calculate_amd, calculate_minimum_distance, calculate_greedy_symmetric_amd, calculate_optimal_symmetric_amd

def to_torch(x, ref):
    return torch.as_tensor(x, device=ref.device, dtype=ref.dtype)

def compute_metrics(embeddings, definition_embeddings, words, spaces, space_functions, show_lda=False):
    # remove suffixes from words
    words = [w.split('_')[0] for w in words]

    # one row per word; columns for each metric-space combo
    columns = ['words'] + [f'{metric}_{space}' for space in spaces for metric in ['amd1','amd2']]
    results = pd.DataFrame({'words': words})
    for col in columns:
        if col != 'words':
            results[col] = None

    # main loop: per word, compute all spaces and fill one row
    for i, word_usage_embeddings in enumerate(embeddings):
        word = words[i]
        word_definition_embeddings = definition_embeddings[word]
        num_defs = len(word_definition_embeddings)

        # split once in the "full" space, then space functions can transform
        halfway = word_usage_embeddings.shape[0] // 2
        usage_embs_t1 = word_usage_embeddings[:halfway].squeeze()
        usage_embs_t2 = word_usage_embeddings[halfway:].squeeze()

        # stack defs once
        def_emb = torch.stack([to_torch(e, word_usage_embeddings) for e in word_definition_embeddings]).squeeze(1)

        # context object so space fns can access whatever they need
        ctx = {
            "word": word,
            "num_defs": num_defs,
            "word_usage_embeddings": word_usage_embeddings,
            "def_emb": def_emb,
        }

        for space, space_fn in zip(spaces, space_functions):
            t1_s, t2_s = space_fn(usage_embs_t1, usage_embs_t2, ctx)
            # t1_s = torch.nn.functional.normalize(t1_s, dim=1)
            # t2_s = torch.nn.functional.normalize(t2_s, dim=1)

            # amd1 = calculate_minimum_distance(t1_s, t2_s, 1)
            # amd2 = calculate_minimum_distance(t1_s, t2_s, 2)

            # results.loc[i, f'amd1_{space}'] = amd1
            # results.loc[i, f'amd2_{space}'] = amd2

            # # compute the mean of AMD1 and AMD2
            # results.loc[i, f'amd_mean_{space}'] = (amd1 + amd2) / 2

            # # min of AMD1 and AMD2
            # results.loc[i, f'amd_min_{space}'] = min(amd1, amd2)

            # # max of AMD1 and AMD2
            # results.loc[i, f'amd_max_{space}'] = max(amd1, amd2)

            # # abs(AMD1-AMD2)
            # results.loc[i, f'amd_diff_{space}'] = abs(amd1 - amd2)

            # # harmonic mean of AMD1 and AMD2
            # if amd1 + amd2 > 0:
            #     results.loc[i, f'amd_hmean_{space}'] = 2 * (amd1 * amd2) / (amd1 + amd2)
            # else:
            #     results.loc[i, f'amd_hmean_{space}'] = 0
            
            apd = calculate_apd(t1_s, t2_s)
            prt = calculate_prt(t1_s, t2_s)
            amd = calculate_amd(t1_s, t2_s)
            amd_sym = calculate_greedy_symmetric_amd(t1_s, t2_s)
            amd_opt_sym = calculate_optimal_symmetric_amd(t1_s, t2_s)

            results.loc[i, f'apd_{space}'] = apd
            results.loc[i, f'prt_{space}'] = prt
            results.loc[i, f'amd_{space}'] = amd
            results.loc[i, f'amd_sym_{space}'] = amd_sym
            results.loc[i, f'amd_opt_sym_{space}'] = amd_opt_sym
    return results

/Users/acw747/Projects/definition_projection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os, time, torch
from spaces import return_full_space, def_space, pca_space, pca_space_num, random_dim_selection, random_dim_selection_num
from embed_defs_functions import compute_spearman_correlation
from def_proj_functions import load_semeval_df, \
        lang_specific_model_names, lang_specific_models, models, definition_models, languages

from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

PATH_TO_DEF_EMBS = os.getenv('PATH_TO_DEF_EMBS')
PATH_TO_USAGE_EMBS = os.getenv('PATH_TO_USAGE_EMBS')


MASTER_CSV = './results/master_results_spaces.csv'

def ensure_csv_header(path=MASTER_CSV):
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, 'w') as f:
            f.write("language,encoder,definition_gen_model,metric,space,spearman,timestamp,other\n")

def append_row_csv(language, encoder_name, def_model, metric, space, value, timestamp, other=""):
    ensure_csv_header()
    with open(MASTER_CSV, 'a') as f:
        f.write(f"{language},{encoder_name},{def_model},{metric},{space},{value},{timestamp},{other}\n")

def run_exp(spaces, space_functions):
    rows = []
    i = 0
    for language in languages:
        print(f"[[Processing language]]: {language}")
        df = load_semeval_df(language)
        # choose encoders the same way as before
        lang_encoders = models + lang_specific_models.get(language, [])

        for model in lang_encoders:
            # print(f"[Using encoder]: {model}")
            # tok, encoder = get_wordtransformer_model(model)

            # normalize encoder name consistently across both pipelines
            if '/' in model:
                encoder_name = lang_specific_model_names[model]
            else:
                encoder_name = model

            # load precomputed usage embeddings (your second loop did this)
            usage_path = f'{PATH_TO_USAGE_EMBS}/{language}_embeddings_by_{encoder_name}.pt'
            embeddings = torch.load(usage_path)

            for def_model in definition_models[:1]:  # == definition_gen_models in your second loop
                i += 1
                defs_dir = f'{PATH_TO_DEF_EMBS}/embeddings/{language}'
                os.makedirs(defs_dir, exist_ok=True)

                defs_path = f'{defs_dir}/{def_model}_definitions_embedded_by_{encoder_name}.pt'

                # 1) Load precomputed definition embeddings
                definition_embeddings = torch.load(defs_path)

                # 2) Compute metrics + scores (retain functionality of second loop)
                results = compute_metrics(embeddings, definition_embeddings, df['words'].tolist(), spaces=spaces, space_functions=space_functions)
                scores = compute_spearman_correlation(df, results, metric_cols=[f'{metric}_{space}' for space in spaces for metric in ['prt','apd','amd','amd_sym','amd_opt_sym']])


                timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())

                for (metric, space), value in scores.items():
                    row = {
                        "language": language,
                        "encoder": encoder_name,
                        "definition_gen_model": def_model,
                        "metric": metric,
                        "space": space,
                        "spearman": value,
                        "timestamp": timestamp,
                        "other": "",
                    }
                    rows.append(row)
                    append_row_csv(language, encoder_name, def_model, metric, space, value, timestamp, other="")




In [ ]:
print("Running experiments...")

run_exp(
    ['full','pca','def','rand'],
    [
        return_full_space,
        pca_space,
        def_space,
        random_dim_selection,
    ]
)



In [ ]:
# Stress test of reducing dimensions

# num_dimensions = [1024, 512, 256, 128, 64, 32, 16, 8, 4, 2, 1]

# for num_dim in num_dimensions:
#     print(f"Running experiments with {num_dim} dimensions...")
#     run_exp(
#         [f'random_num{num_dim}',f'pca_num{num_dim}'],
#         [
#             partial(random_dim_selection_num, num_dims=num_dim),
#             partial(pca_space_num, num_components=num_dim),
#         ]
    # )